# Cache RegionCLIP 10k TRAIN + 1k VALID
Chọn 1 GPU T4, bật Internet. Gắn checkpoint RN50 đã chạy v135, csa-prompt-full-karpathy-v175 và mscoco-2014. Giữ toàn bộ thư mục regionclip_slot_cache10k; gửi ZIP review nhỏ.


In [ ]:
from pathlib import Path
import os, subprocess, sys, tempfile, zipfile
CHECKPOINT_PATH = ''  # Optional full path to the mounted official RN50 checkpoint.
FILENAMES = ('regionclip_pretrained-cc_rn50.pth', 'regionclip_pretrained-cc_rn50.bin')
REVISION = '4b8513b56e24827e3d6468e1f2105869f35c2d0b'
if CHECKPOINT_PATH:
    CHECKPOINT = Path(CHECKPOINT_PATH)
    if not CHECKPOINT.is_file() or CHECKPOINT.name not in FILENAMES:
        raise FileNotFoundError(f'Expected an RN50 checkpoint at {CHECKPOINT}')
else:
    candidates = sorted(path for name in FILENAMES for path in Path('/kaggle/input').rglob(name) if path.is_file())
    if not candidates:
        raise FileNotFoundError(f'Attach a Kaggle Dataset containing {FILENAMES[0]} or {FILENAMES[1]} via Add Input, then rerun this cell.')
    if len(candidates) > 1:
        raise ValueError('Set CHECKPOINT_PATH to one of these matches: '+', '.join(map(str,candidates)))
    CHECKPOINT = candidates[0]
print('RegionCLIP checkpoint:', CHECKPOINT, flush=True)
PROMPT_ROOT=Path('/kaggle/input/datasets/ducanh2403/csa-prompt-full-karpathy-v175')
IMAGE_ROOT=Path('/kaggle/input/datasets/vuthetam/mscoco-2014/images')
if not PROMPT_ROOT.is_dir() or not IMAGE_ROOT.is_dir():
    raise FileNotFoundError('Attach v175 prompt links and MS COCO. Adjust mount roots if needed.')
OUTPUT = Path('/kaggle/working/regionclip_slot_cache10k')
OUTPUT.mkdir(exist_ok=False)
ROOT=Path(tempfile.mkdtemp(prefix='regionclip-fixed-roi-'))
REPO=ROOT/'RegionCLIP'
DEPS=ROOT/'deps'
env=dict(os.environ,MAX_JOBS='2',CUDA_VISIBLE_DEVICES='')
def run(args, timeout, environment=None, cwd=None):
    with (OUTPUT/'notebook.log').open('a',encoding='utf8') as log:
        p=subprocess.Popen([str(x) for x in args],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                           text=True,encoding='utf8',errors='replace',env=environment,cwd=cwd)
        try:
            result, _=p.communicate(timeout=timeout)
        except subprocess.TimeoutExpired:
            p.kill()
            result, _=p.communicate()
            log.write(result)
            raise
        log.write(result)
        print(result[-6000:])
        if p.returncode:
            raise subprocess.CalledProcessError(p.returncode,args)


In [ ]:
WORKER_SOURCE='REVISION=\'4b8513b56e24827e3d6468e1f2105869f35c2d0b\'\nimport hashlib\nimport json\nfrom pathlib import Path\nimport sys\nimport time\n\nimport numpy as np\nimport torch\nfrom detectron2.config import get_cfg\nfrom detectron2.data import detection_utils as utils\nfrom detectron2.data import transforms as T\nfrom detectron2.modeling import build_backbone\nfrom detectron2.structures import Boxes, ImageList\nfrom detectron2.modeling.poolers import ROIPooler\n\nrepo, checkpoint, out, prompt_root, image_root = map(Path,sys.argv[1:])\nif not torch.cuda.is_available():\n    raise RuntimeError(\'Select a CUDA GPU\')\ndevice=torch.device(\'cuda:0\')\ncfg=get_cfg()\ncfg.merge_from_file(str(repo/\'configs/LVISv1-InstanceSegmentation/CLIP_fast_rcnn_R_50_C4_zsinf.yaml\'))\nbackbone=build_backbone(cfg)\nif not hasattr(backbone,\'attnpool\'):\n    raise ValueError(\'Expected RegionCLIP RN50 attention pooling\')\nwith checkpoint.open(\'rb\') as source:\n    checkpoint_sha=hashlib.file_digest(source,\'sha256\').hexdigest()\nif checkpoint_sha!=\'afbc58d7d4bd96e2b10cc0d1928a912ed519c3b3215df7e5f3285e4bb09c7c58\':\n    raise ValueError(\'RegionCLIP checkpoint differs from V4\')\nlinks,=prompt_root.rglob(\'prompt_links.jsonl\')\nwith links.open(\'rb\') as stream:\n    digest=hashlib.file_digest(stream,\'sha256\').hexdigest()\nif digest!=\'e90c79b3b04540630b4bd2d279a4070ef05da7f4b65970c952851c37e2423bf0\':\n    raise ValueError(\'Unexpected v175 prompt links\')\nlimits={\'train\':10000,\'val\':1000}\npools={group:[] for group in limits}\nwith links.open(encoding=\'utf8\') as stream:\n    for line in stream:\n        row=json.loads(line)\n        group=row[\'group\']\n        if group in pools:\n            key=hashlib.sha256(f"regionclip-cache42:{group}:{row[\'coco_id\']}".encode()).digest()\n            pools[group].append((key,row))\ncases=[]\nfor group,count in limits.items():\n    picked=sorted(pools[group],key=lambda pair:pair[0])[:count]\n    if len(picked)!=count:\n        raise ValueError(\'Insufficient eligible v175 images for \'+group)\n    cases.extend(row for _,row in picked)\nif len({row[\'coco_id\'] for row in cases})!=sum(limits.values()):\n    raise ValueError(\'Duplicate image ID\')\npayload=torch.load(checkpoint,map_location=\'cpu\',weights_only=False)\nstate=payload[\'model\']\nweights={}\nfor key,value in state.items():\n    key=key.removeprefix(\'module.\')\n    if key.startswith(\'backbone.\'):\n        weights[key.removeprefix(\'backbone.\')]=torch.as_tensor(value)\nbackbone.load_state_dict(weights,strict=True)\ndel payload,state,weights\nbackbone.to(device).eval()\npooler=ROIPooler(output_size=cfg.MODEL.ROI_BOX_HEAD.POOLER_RESOLUTION,\n                scales=(1/backbone.output_shape()[\'res4\'].stride,),\n                sampling_ratio=cfg.MODEL.ROI_BOX_HEAD.POOLER_SAMPLING_RATIO,\n                pooler_type=cfg.MODEL.ROI_BOX_HEAD.POOLER_TYPE).to(device)\nmean=torch.tensor(cfg.MODEL.PIXEL_MEAN,device=device)[:,None,None]\nstd=torch.tensor(cfg.MODEL.PIXEL_STD,device=device)[:,None,None]\nif max(cfg.MODEL.PIXEL_MEAN)>=1:\n    raise ValueError(\'Unexpected image normalization\')\ndef digest_file(path):\n    with path.open(\'rb\') as stream:\n        return hashlib.file_digest(stream,\'sha256\').hexdigest()\n\ndef save_json(path,value):\n    tmp=path.with_suffix(path.suffix+\'.tmp\')\n    tmp.write_text(json.dumps(value,indent=2,allow_nan=False)+\'\\n\',encoding=\'utf8\')\n    tmp.replace(path)\n\ncohort={group:[row[\'coco_id\'] for row in cases if row[\'group\']==group] for group in limits}\ncohort_path=out/\'cohort.json\'\nsave_json(cohort_path,cohort)\narrays={}\nrecords=[]\nshards=[]\nshard_size=500\nstarted=time.monotonic()\nwith torch.inference_mode():\n    for index,case in enumerate(cases,1):\n        cid=case[\'coco_id\']\n        filename=case[\'filename\']\n        if filename not in (f\'COCO_train2014_{cid:012d}.jpg\',f\'COCO_val2014_{cid:012d}.jpg\'):\n            raise ValueError(f\'Unexpected COCO filename for {cid}\')\n        rois=case[\'objects\']\n        if rois:\n            path=image_root/filename.split(\'_\')[1]/filename\n            with path.open(\'rb\') as stream:\n                image_sha=hashlib.file_digest(stream,\'sha256\').hexdigest()\n            image=utils.read_image(str(path),format=cfg.INPUT.FORMAT)\n            augmented=T.AugInput(image)\n            transform=T.AugmentationList(utils.build_augmentation(cfg,False))(augmented)\n            h,w=image.shape[:2]\n            raw=np.asarray([row[\'roi_bbox\'] for row in rois],dtype=np.float64)*[w,h,w,h]\n            if not np.isfinite(raw).all() or not (raw[:,2:]>raw[:,:2]).all():\n                raise ValueError(\'Invalid YOLO boxes\')\n            shifted=transform.apply_box(raw)\n            x=torch.as_tensor(np.ascontiguousarray(augmented.image.transpose(2,0,1)),device=device).float()\n            batch=ImageList.from_tensors([(x/255-mean)/std],backbone.size_divisibility)\n            spatial=backbone(batch.tensor)[\'res4\']\n            boxes=Boxes(torch.as_tensor(shifted,dtype=torch.float32,device=device))\n            vector=backbone.attnpool(backbone.layer4(pooler([spatial],[boxes])))\n            if vector.ndim!=2 or tuple(vector.shape)!=(len(rois),1024) or not torch.isfinite(vector).all():\n                raise ValueError(\'Unexpected RegionCLIP RN50 vectors\')\n            if (vector.norm(dim=1)<=0).any():\n                raise ValueError(\'Degenerate vectors\')\n            arrays[str(cid)]=vector.float().cpu().numpy()\n        else:\n            image_sha=None\n            arrays[str(cid)]=np.empty((0,1024),np.float32)\n        records.append(dict(coco_id=cid,group=case[\'group\'],filename=filename,\n                            image_sha256=image_sha,feature_shape=list(arrays[str(cid)].shape),\n                            slots=[dict(instance_id=r[\'instance_id\'],roi=r[\'roi\'],\n                                        bbox=r[\'roi_bbox\'],text=r[\'text\']) for r in rois]))\n        if index%shard_size==0 or index==len(cases):\n            stem=f\'shard_{len(shards):03d}\'\n            path=out/(stem+\'.npz\')\n            temporary=out/(stem+\'.npz.tmp\')\n            with temporary.open(\'wb\') as stream:\n                np.savez(stream,**arrays)\n            temporary.replace(path)\n            rows_path=out/(stem+\'.json\')\n            save_json(rows_path,records)\n            shards.append(dict(vectors=path.name,metadata=rows_path.name,\n                               vectors_sha256=digest_file(path),metadata_sha256=digest_file(rows_path),\n                               images=len(records),regions=sum(len(r[\'slots\']) for r in records),\n                               without_slots=sum(not r[\'slots\'] for r in records)))\n            save_json(out/\'progress.json\',dict(images_done=index,shards=shards))\n            print(f\'RegionCLIP cached {index}/{len(cases)}\',flush=True)\n            arrays.clear()\n            records.clear()\nif sum(s[\'images\'] for s in shards)!=sum(limits.values()):\n    raise ValueError(\'Incomplete shard coverage\')\nresult=dict(status=\'COMPLETE\',checkpoint_sha256=checkpoint_sha,source_revision=REVISION,\n            prompt_links_sha256=digest,cohort_sha256=digest_file(cohort_path),images=limits,\n            regions=sum(s[\'regions\'] for s in shards),without_slots=sum(s[\'without_slots\'] for s in shards),\n            extraction_seconds=time.monotonic()-started,shards=shards)\nsave_json(out/\'result.json\',result)\n'
WORKER = ROOT/'run.py'
WORKER.write_text(WORKER_SOURCE,encoding='utf8')
try:
    run(['git','init','-q',REPO],30)
    run(['git','-C',REPO,'fetch','--depth','1','https://github.com/microsoft/RegionCLIP.git',REVISION],180)
    run(['git','-C',REPO,'checkout','--detach',REVISION],60)
    run([sys.executable,'-m','pip','install','--target',DEPS,
         'numpy>=1.26,<2.3','yacs','fvcore','iopath','termcolor','diffdist','ftfy','pycocotools','omegaconf'],240)
    setup_path=REPO/'setup.py'
    setup_source=setup_path.read_text(encoding='utf8')
    version_option='version=get_version(),'
    if setup_source.count(version_option)!=1:
        raise RuntimeError('RegionCLIP setup version field differs from the pinned source')
    setup_path.write_text(setup_source.replace(version_option,'version="0.0.0",'),encoding='utf8')
    transform_path=REPO/'detectron2/data/transforms/transform.py'
    transform_source=transform_path.read_text(encoding='utf8')
    old_default='interp=Image.LINEAR,'
    if transform_source.count(old_default)!=1:
        raise RuntimeError('RegionCLIP Pillow default differs from the pinned source')
    transform_path.write_text(transform_source.replace(
        old_default,'interp=Image.Resampling.BILINEAR,'),encoding='utf8')
    for module in ('rcnn.py','clip_rcnn.py'):
        module_path=REPO/'detectron2/modeling/meta_arch'/module
        module_source=module_path.read_text(encoding='utf8')
        old_import='from numpy.lib import pad'
        if module_source.count(old_import)!=1:
            raise RuntimeError(f'Unexpected NumPy import in {module}')
        module_path.write_text(module_source.replace(old_import,'from numpy import pad'),encoding='utf8')
    env['PYTHONPATH']=str(DEPS)
    run([sys.executable,'setup.py','build_ext','--inplace'],480,env,REPO)
    inference_env=dict(os.environ,PYTHONPATH=os.pathsep.join((str(REPO),str(DEPS))))
    run([sys.executable,WORKER,REPO,CHECKPOINT,OUTPUT,PROMPT_ROOT,IMAGE_ROOT],5400,inference_env)
finally:
    review=OUTPUT.with_name(OUTPUT.name+'_review.zip')
    with zipfile.ZipFile(review,'w',zipfile.ZIP_DEFLATED) as z:
        for name in ('notebook.log','result.json','progress.json','cohort.json'):
            if (OUTPUT/name).exists():
                z.write(OUTPUT/name,name)
    print('SEND REVIEW',review)
